# reshape-back — worked example 1: reshape_back for a 3-D flatten

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `reshape-back`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Reshape only rearranges which index maps to which storage slot; it is a pure permutation, so its Jacobian is a permutation matrix. The backward of `out = x.reshape(new_shape)` is therefore the inverse reshape: `grad_out.reshape(x.shape)`. The original `x.shape` is the authoritative target, not the forward's `new_shape`.

## Worked solution

The forward flattens a `(2, 3, 4)` tensor to `(24,)`. In `reshape_back(grad_out, out, x, new_shape)` we ignore `new_shape` and reshape the incoming gradient straight back to `x.shape`, because every output element corresponds to exactly one input element and the gradient just needs to land back in the original layout. We verify against autograd: build a leaf `x` with `requires_grad`, run the same flatten, backward a known upstream gradient, and confirm `x.grad` matches our hand-rolled `reshape_back`. The shapes line up because reshape preserves element count.

In [ ]:
Tensor = t.Tensor


def reshape_back(grad_out: Tensor, out: Tensor, x: Tensor, new_shape: tuple) -> Tensor:
    # Reshape is a storage permutation; backward = inverse reshape to x.shape.
    return grad_out.reshape(x.shape)


t.manual_seed(0)
x = t.randn(2, 3, 4)
out = x.reshape(24)
grad_out = t.arange(24, dtype=t.float32)
grad_x = reshape_back(grad_out, out, x, (24,))
print('grad_x shape:', tuple(grad_x.shape))

# cross-check against autograd
xg = x.clone().requires_grad_(True)
xg.reshape(24).backward(grad_out)
print('matches autograd:', t.allclose(grad_x, xg.grad))